<a href="https://colab.research.google.com/github/Jason-fy-wang/LLM_agent/blob/main/huggface_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print()

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [1]:
!pip install torch torchvision transformers accelerate

In [3]:
!pip install -U Qwen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.7/522.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
  Attempting uninstall: scikit-learn
    Found existi

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# load model
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, dtype=torch.float16)

model.eval()
print("model load done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model load done


In [2]:
def build_prompt(user_input:str) -> str :
  return f"请解释中文成语的含义与出处: {user_input}"

In [6]:
def run_inference(prompt: str, max_tokens: int=128, temperature: float=0.7):
  input = tokenizer(prompt, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
        **input,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
    )
  decoded = tokenizer.decode(outputs[0], skip_special_tokens = True)
  return decoded


In [7]:
idioms = ["画蛇添足","纸上谈兵","掩耳盗铃","望梅止渴","指鹿为马","多多益善"]
print(">>> begin batch job")

for idiom in idioms:
  prompt = build_prompt(idiom)
  print(f"prompt: {prompt}")
  result = run_inference(prompt)
  print(f"result: {result}")

>>> begin batch job
prompt: 请解释中文成语的含义与出处: 画蛇添足
result: 请解释中文成语的含义与出处: 画蛇添足
成语：画蛇添足

含义：画蛇添足比喻做了多余的事，反而不合适，弄巧成拙。

出处：《战国策·齐策二》

原文：蛇固无足，子安能为之足？”

故事背景：战国时期，齐国的谋士淳于髡（kūn）以善辩著称。一次，他奉齐王之命出使楚国，为了活跃气氛，他带来了一些酒，邀请宾客们一起饮酒。饮酒时，淳于髡提议进行一个游戏：谁先画完蛇，谁就能得到酒。宾客
prompt: 请解释中文成语的含义与出处: 纸上谈兵
result: 请解释中文成语的含义与出处: 纸上谈兵

成语“纸上谈兵”用来形容人只凭书本知识或理论去空谈，缺乏实际经验或实践能力。这个成语出自《史记·廉颇蔺相如列传》。故事讲的是战国时期，赵国名将赵奢的儿子赵括，从小喜欢谈论军事，以兵法为乐，但缺乏实战经验。后来赵国在长平之战中，赵王任命赵括为统帅，结果赵括纸上谈兵，导致赵军大败，损失惨重。

这个成语常用来批评那些不切实际、脱离实际的人，也提醒人们
prompt: 请解释中文成语的含义与出处: 掩耳盗铃
result: 请解释中文成语的含义与出处: 掩耳盗铃
成语“掩耳盗铃”的含义是：比喻自己欺骗自己，明明掩盖不住的事情偏要想法掩盖。这个成语出自《吕氏春秋·自知》。
出处：
《吕氏春秋·自知》原文：“有献鸡卵于鲁哀公者，曰：‘此鸡子之贵也。’哀公曰：‘寡人闻之，鸟有母子，而人无母子。’对曰：‘此鸡子之贵也。’哀公曰：‘寡人闻之，鸟有母子，而人无母子。’对曰
prompt: 请解释中文成语的含义与出处: 望梅止渴
result: 请解释中文成语的含义与出处: 望梅止渴
成语：望梅止渴

“望梅止渴”是一个汉语成语，其字面意思是“看着梅子就能解渴”，用来比喻用空想或假象来安慰自己，以缓解现实中的困难或痛苦。

### 出处：
这个成语出自《世说新语·假谲》。故事讲的是三国时期，曹操在行军途中，士兵们口渴，曹操便说：“前面有梅林。”士兵们听到后，嘴里都流出了口水，从而暂时解除了口渴的感觉。后来，人们用“望梅止渴”来形容用想象来安慰
prompt: 请解释中文成语的含义与出处: 指鹿为马
result: 请解释中文成语的含义与出处: 指鹿为马
成语“指鹿为马”是一个常用的中文成语，